**udp_receiver.ipynb**

This notebook contains the UDP socket functions for the Receiver PYNQ board.

Purpose: 
receiver listens for incoming UDP packets transmitted from the **Sender PYNQ-Z2 board**, which processes sensor data (i.e. temperature and microphone signals). Each packet contains a comma-delimited string representing the measured sensor values and associated status information.

functions in this notebook perform the following tasks:

- Create and bind a **UDP socket** to a specified IP address and port.
- Receive incoming sensor data packets over the network.
- Parse the received data into structured variables including:
  - Temperature in Celsius and Fahrenheit
  - Processed microphone signal level
  - Status indicators generated by the sender system

In [1]:
import socket

def create_udp_receiver(local_ip="0.0.0.0", port=12345):
    """
    Create and bind a UDP receiver socket.

    Parameters:
        local_ip (str): IP address to bind to. Use 0.0.0.0 to listen on all interfaces.
        port (int): UDP port number.

    Returns:
        sock: bound UDP socket
    """
    sock = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
    sock.bind((local_ip, port))
    return sock

In [2]:
def receive_packet(sock, buffer_size=1024):
    """
    Receive one UDP packet and parse it.

    Expected format:
        temp_c,temp_f,mic_level,temp_status,mic_status

    Example:
        23.50,74.30,312.40,NORMAL,QUIET

    Returns:
        payload (dict): parsed sensor values
        addr (tuple): sender IP and port
    """
    data, addr = sock.recvfrom(buffer_size)
    message = data.decode().strip()

    parts = message.split(",")

    if len(parts) < 5:
        raise ValueError(f"Invalid packet format: {message}")

    payload = {
        "temp_c": float(parts[0]),
        "temp_f": float(parts[1]),
        "mic_level": float(parts[2]),
        "temp_status": parts[3].strip(),
        "mic_status": parts[4].strip()
    }

    return payload, addr

In [5]:
# Optional quick test cell
# Run only when the Sender board is already transmitting

sock = create_udp_receiver("0.0.0.0", 12345)
payload, addr = receive_packet(sock)
print("Received from:", addr)
print(payload)

KeyboardInterrupt: 